In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)


# 01 分箱模块 (core.binning)

基于真实信贷数据 `hscredit.xlsx`，覆盖所有分箱算法和常用参数。

**数据说明**:
- 12448 条贷款样本，85个字段
- 目标变量: `FPD15`（首期逾期15天坏样本标记）
- 外部评分: `bj_qy24` / `mobtech80` / `bairong_v1` 等
- 行为特征: 网贷次数/金额/逾期记录等 70+ 个特征

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import hscredit as hsc
from hscredit import init_setting, OptimalBinning
from hscredit import (
    UniformBinning, QuantileBinning, TreeBinning, ChiMergeBinning,
    BestKSBinning, BestIVBinning, MDLPBinning, CartBinning,
    KMeansBinning, MonotonicBinning, SmoothBinning,
    BestLiftBinning, TargetBadRateBinning,
)
init_setting()

# 加载真实信贷数据
df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
target = 'FPD15'
features = [c for c in df.columns if c not in ['MOB1','MOB2','loan_date','loan_month','FPD15','SFPD15']]
print(f'样本数: {len(df):,}, 坏率: {df[target].mean():.2%}, 特征数: {len(features)}')

## 1. 快速开始 — 单特征分箱

In [ ]:
feat = 'bj_qy24'
binner = OptimalBinning(method='quantile', max_n_bins=8)
binner.fit(df[[feat]], df[target])
bt = binner.bin_tables_[feat]
print(f'{feat} - IV: {bt["分档IV值"].sum():.4f}')
bt[['分箱标签','样本总数','坏样本率','分档WOE值','分档IV值']]

bj_qy24 - IV: 0.0763


,分箱标签,样本总数,坏样本率,分档WOE值,分档IV值
0,"(-inf, 543.0]",2480,0.0895,0.3245,0.0242
1,"(543.0, 581.0]",2489,0.0775,0.1678,0.0061
2,"(581.0, 615.0]",2484,0.0684,0.0331,0.0002
3,"(615.0, 655.0]",2503,0.0543,-0.2127,0.0083
4,"(655.0, +inf]",2492,0.0421,-0.4798,0.0376


## 2. 所有分箱方法对比

In [ ]:
feat = 'bj_qy24'
X1 = df[[feat]].dropna(); y1 = df.loc[X1.index, target]
X1 = X1.reset_index(drop=True); y1 = y1.reset_index(drop=True)
methods = {
    'uniform': UniformBinning(max_n_bins=8),
    'quantile': QuantileBinning(max_n_bins=8),
    'tree': TreeBinning(max_n_bins=8),
    'chi': ChiMergeBinning(max_n_bins=8),
    'best_ks': BestKSBinning(max_n_bins=8),
    'best_iv': BestIVBinning(max_n_bins=8),
    'mdlp': MDLPBinning(max_n_bins=8),
    'cart': CartBinning(max_n_bins=8),
    'kmeans': KMeansBinning(max_n_bins=8),
    'smooth': SmoothBinning(max_n_bins=8),
    'best_lift': BestLiftBinning(max_n_bins=8),
    'target_bad_rate': TargetBadRateBinning(max_n_bins=8),
    'monotonic': MonotonicBinning(max_n_bins=8, monotonic='auto'),
}
rows = []
for name, b in methods.items():
    try:
        b.fit(X1, y1)
        bt = b.bin_tables_.get(feat, pd.DataFrame())
        iv_val = bt['分档IV值'].sum() if '分档IV值' in bt.columns else 0
        rows.append({'方法': name, '分箱数': len(bt), 'IV': round(iv_val,4), '最大坏率': round(bt['坏样本率'].max(),4) if '坏样本率' in bt.columns else 0})
    except Exception as e:
        rows.append({'方法': name, '分箱数': 0, 'IV': 0, '最大坏率': str(e)[:40]})
pd.DataFrame(rows).sort_values('IV', ascending=False)

,方法,分箱数,IV,最大坏率
4,best_ks,8,0.1210,0.1373
6,mdlp,8,0.1186,0.2262
5,best_iv,8,0.1178,0.1264
7,cart,8,0.1131,0.1825
9,smooth,8,0.1078,0.1183
2,tree,8,0.1071,0.1141
3,chi,8,0.1071,0.1071
12,monotonic,8,0.0968,0.1158
8,kmeans,8,0.0837,0.1044
10,best_lift,5,0.0796,0.1096


## 3. 单调性约束分箱（外部评分）

In [ ]:
# 外部评分通常应该单调：分越高坏率越低
external_scores = ['bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3']
for score_feat in external_scores:
    valid = df[[score_feat]].dropna()
    y_v = df.loc[valid.index, target].reset_index(drop=True)
    X_v = valid.reset_index(drop=True)
    try:
        b = OptimalBinning(method='monotonic', monotonic='descending', max_n_bins=8)
        b.fit(X_v, y_v)
        bt = b.bin_tables_[score_feat]
        rates = bt['坏样本率'].round(4).tolist()
        print(f'{score_feat:20s} IV={bt["分档IV值"].sum():.4f}  坏率趋势: {rates}')
    except Exception as e:
        print(f'{score_feat}: {e}')

bj_qy24              IV=0.0968  坏率趋势: [0.1158, 0.0841, 0.0827, 0.0665, 0.0559, 0.0526, 0.0491, 0.0354]
mobtech80            IV=0.0803  坏率趋势: [0.0867, 0.0803, 0.0733, 0.0483]
bairong_v1           IV=0.2922  坏率趋势: [0.1253, 0.0884, 0.0784, 0.059, 0.0514, 0.0449, 0.03, 0.0112]
xiaoniu_c3           IV=0.1780  坏率趋势: [0.1063, 0.1, 0.0748, 0.0565, 0.0421, 0.0302, 0.0158]


## 4. 批量分箱 — 行为特征

In [ ]:
behavior_feats = [f for f in features if any(k in f for k in ['count','sum','amt','loan_count','lender'])][:12]
batcher = OptimalBinning(method='mdlp', max_n_bins=6)
batcher.fit(df[behavior_feats], df[target])
iv_rows = []
for feat, bt in batcher.bin_tables_.items():
    if '分档IV值' in bt.columns:
        iv_rows.append({'特征': feat, 'IV': round(bt['分档IV值'].sum(),4), '箱数': len(bt)})
pd.DataFrame(iv_rows).sort_values('IV', ascending=False)

,特征,IV,箱数
5,consumer_finance_lender_count_12m,0.1977,6
11,consumer_finance_loan_lender_count,0.1975,6
6,consumer_finance_lender_count_24m,0.1879,6
8,consumer_finance_loan_credit_line,0.0853,6
10,consumer_finance_loan_credit_line_max,0.0719,6
4,charging_fail_count_6m,0.0642,6
9,consumer_finance_loan_credit_line_avg,0.0566,6
7,consumer_finance_loan_confidence,0.0563,6
0,charging_fail_count_12m,0.0491,6
3,charging_fail_count_3m,0.0462,6


## 5. 自定义分割点

In [ ]:
b_custom["bcf_fpv1"]

TypeError: 'OptimalBinning' object is not subscriptable

In [ ]:
# 业务规则：征信评分按固定切点分箱
b_custom = OptimalBinning()
b_custom.fit(df[['bcf_fpv1']], df[target])
b_custom.bin_tables_['bcf_fpv1'][['分箱标签','样本总数','样本占比','坏样本率','分档IV值']]

,分箱标签,样本总数,样本占比,坏样本率,分档IV值
0,"(-inf, 127.0]",189,0.0152,0.0529,0.0008
1,"(127.0, 127.5]",5,0.0004,0.6000,0.0106
2,"(127.5, 242.0]",733,0.0589,0.0259,0.0378
3,"(242.0, 294.0]",659,0.0529,0.0091,0.1001
4,"(294.0, +inf]",678,0.0545,0.0015,0.2210
5,missing,10184,0.8181,0.0773,0.0237


## 6. 分箱可视化

In [ ]:
from hscredit import bin_plot
b_vis = OptimalBinning(method='mdlp', max_n_bins=8)
b_vis.fit(df[['bj_qy24']], df[target])
fig = bin_plot(b_vis.bin_tables_['bj_qy24'], desc='bj_qy24 征信评分')
plt.show()

## 7. WOE 转换

In [ ]:
top_feats = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m']
b_woe = OptimalBinning(method='mdlp', max_n_bins=6)
b_woe.fit(df[top_feats].fillna(-999), df[target])
X_woe = b_woe.transform(df[top_feats].fillna(-999), metric='woe')
print('WOE转换后:')
print(X_woe.head(3))
print('\n各特征WOE取值范围:')
for c in X_woe.columns:
    print(f'  {c}: [{X_woe[c].min():.3f}, {X_woe[c].max():.3f}]')

WOE转换后:
   bj_qy24  mobtech80  bairong_v1  lender_count_12m  overdue_loan_m1_count_12m
0  -0.1859    -0.3368     -2.5073           -0.3178                     1.2578
1   0.1807    -0.1285     -0.2625            0.2480                     1.2578
2  -0.1859    -0.3368     -0.2625           -0.1491                     1.2578

各特征WOE取值范围:
  bj_qy24: [-0.619, 1.414]
  mobtech80: [-0.337, 0.397]
  bairong_v1: [-2.507, 3.743]
  lender_count_12m: [-1.742, 0.248]
  overdue_loan_m1_count_12m: [0.000, 1.258]
